#### Importing Libraries

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

#### Loading the Dataset

In [3]:
data_path = Path("../data/raw/20000-Utterances-Training-dataset-for-chatbots-virtual-assistant-Bitext-sample.csv")
df = pd.read_csv(data_path)
print(f"Dataset Shape: {df.shape}")
display(df.head())

Dataset Shape: (21534, 4)


,flags,utterance,category,intent
0,BILC,"I don't have an online account, what do I have...",ACCOUNT,create_account
1,BILQZ,can you tell me if i can regisger two accounts...,ACCOUNT,create_account
2,BPLC,"I have no online account, open one, please",ACCOUNT,create_account
3,BIPLD,"could you ask an agent how to open an account,...",ACCOUNT,create_account
4,BLQC,"i want an online account, create one",ACCOUNT,create_account


#### Inspecting Dataset Structure

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 21534 entries, 0 to 21533
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   flags      21534 non-null  str  
 1   utterance  21534 non-null  str  
 2   category   21534 non-null  str  
 3   intent     21534 non-null  str  
dtypes: str(4)
memory usage: 2.3 MB


#### Identifying Unique Values

In [5]:
unique_counts = df.nunique().reset_index()
unique_counts.columns = ["Column", "Unique Values"]
display(unique_counts)

,Column,Unique Values
0,flags,511
1,utterance,21513
2,category,11
3,intent,27


#### Examining Category and Intent Labels

In [6]:
print("Categories:")
print(sorted(df["category"].unique()))
print("\nIntents:")
print(sorted(df["intent"].unique()))

Categories:
['ACCOUNT', 'CANCELLATION_FEE', 'CONTACT', 'DELIVERY', 'FEEDBACK', 'INVOICES', 'NEWSLETTER', 'ORDER', 'PAYMENT', 'REFUNDS', 'SHIPPING']

Intents:
['cancel_order', 'change_order', 'change_shipping_address', 'check_cancellation_fee', 'check_invoices', 'check_payment_methods', 'check_refund_policy', 'complaint', 'contact_customer_service', 'contact_human_agent', 'create_account', 'delete_account', 'delivery_options', 'delivery_period', 'edit_account', 'get_invoice', 'get_refund', 'newsletter_subscription', 'payment_issue', 'place_order', 'recover_password', 'registration_problems', 'review', 'set_up_shipping_address', 'switch_account', 'track_order', 'track_refund']


#### Examining Category-Intent Relationships

In [7]:
category_intent_summary = (
    df.groupby("category")
    .agg(
        Total_Utterances=("utterance", "count"),
        Unique_Intents=("intent", "nunique")
    )
    .sort_values("Total_Utterances", ascending=False)
    .reset_index()
)
display(category_intent_summary)

,category,Total_Utterances,Unique_Intents
0,PAYMENT,4636,2
1,ACCOUNT,4557,6
2,CONTACT,3081,2
3,INVOICES,2443,2
4,ORDER,2256,4
5,REFUNDS,1932,3
6,FEEDBACK,1326,2
7,DELIVERY,501,2
8,CANCELLATION_FEE,360,1
9,NEWSLETTER,236,1


Observations
- PAYMENT (4,636) and ACCOUNT (4,557) collectively represent approximately 42.7% of all utterances, indicating a concentration in these two categories.
- ACCOUNT contains the most distinct intents (6), followed by ORDER (4) and REFUNDS (3), suggesting greater intent diversity within these categories.
- SHIPPING (206) and NEWSLETTER (236) have the fewest utterances, highlighting substantial differences in category representation.

#### Examining Intent Distribution

In [8]:
intent_distribution = (
    df["intent"]
    .value_counts()
    .rename_axis("Intent")
    .reset_index(name="Count")
)
intent_distribution["Percentage"] = (
    intent_distribution["Count"] / len(df) * 100
).round(2)
display(intent_distribution)

,Intent,Count,Percentage
0,payment_issue,4366,20.27
1,create_account,2122,9.85
2,contact_customer_service,2055,9.54
3,get_invoice,1430,6.64
4,track_order,1224,5.68
5,get_refund,1150,5.34
6,contact_human_agent,1026,4.76
7,check_invoices,1013,4.70
8,recover_password,986,4.58
9,change_order,926,4.30


#### Observations
- **`payment_issue` (20.27%)** is the most frequent intent, followed by `create_account` (9.85%) and `contact_customer_service` (9.54%).
- **`cancel_order` (0.17%)** is the least represented intent, indicating substantial class imbalance across the 27 classes.
- **Modeling Decision:** Retain all intent classes and prioritize Macro F1-score and per-class recall during evaluation.

#### Examining Flag Distributions

In [9]:
flag_distribution = (
    df["flags"]
    .value_counts()
    .head(20)
    .rename_axis("Flag")
    .reset_index(name="Count")
)
flag_distribution["Percentage"] = (
    flag_distribution["Count"] / len(df) * 100
).round(2)
display(flag_distribution)

,Flag,Count,Percentage
0,BILC,2752,12.78
1,BILQC,2068,9.60
2,BL,1253,5.82
3,BIL,1042,4.84
4,BLC,967,4.49
5,BLQC,783,3.64
6,BILQ,734,3.41
7,BLQ,608,2.82
8,BILCZ,511,2.37
9,BILQCZ,396,1.84


#### Observations
- **`BILC` (12.78%)** and **`BILQC` (9.60%)** are the most frequent flag combinations, collectively representing 22.38% of utterances.
- The dataset contains **511 unique flag combinations**, indicating considerable variation in the associated metadata.
- **Decision:** Investigate the meaning of these flags before determining whether they provide useful information for analysis or modeling.

#### Examining Flag Associations with Customer Utterances

In [10]:
flag_samples = (
    df.groupby("flags", sort=False)
    .head(2)
    .loc[:, ["flags", "utterance", "intent"]]
    .head(20)
)
display(flag_samples)

,flags,utterance,intent
0,BILC,"I don't have an online account, what do I have...",create_account
1,BILQZ,can you tell me if i can regisger two accounts...,create_account
2,BPLC,"I have no online account, open one, please",create_account
3,BIPLD,"could you ask an agent how to open an account,...",create_account
4,BLQC,"i want an online account, create one",create_account
5,BSLC,"i want an account, i need help opening one",create_account
6,BLZ,tell me if I can register two online accounts...,create_account
7,BPLQ,i want to know if i could create two profiles ...,create_account
8,BIMLQW,can you tell me if i can create more than one ...,create_account
9,BLZ,were to create an onlind account,create_account


#### Observations
- Different flag combinations appear across utterances associated with the same intent, suggesting that flags capture characteristics beyond the intent label.
- Sample utterances contain variations in phrasing, capitalization, spelling, and sentence structure.
- **Decision:** Treat `flags` as auxiliary metadata and exclude it from the initial text classification models unless its meaning and availability during inference are established.

#### Identifying Duplicate Records and Utterances

In [11]:
duplicate_summary = pd.DataFrame({
    "Duplicate Type": [
        "Fully Duplicated Records",
        "Duplicated Utterances",
        "Utterances with Multiple Intents"
    ],
    "Count": [
        df.duplicated().sum(),
        df["utterance"].duplicated().sum(),
        (df.groupby("utterance")["intent"].nunique() > 1).sum()
    ]
})
display(duplicate_summary)

,Duplicate Type,Count
0,Fully Duplicated Records,12
1,Duplicated Utterances,21
2,Utterances with Multiple Intents,0


#### Observations
- **12 fully duplicated records** were identified, representing approximately 0.06% of the dataset.
- **21 duplicated utterances** were found, indicating a small amount of repeated customer text.
- **Decision:** No utterances have conflicting intent labels. Remove duplicate utterances during data preparation to prevent identical queries from appearing in both training and testing sets.

#### Identifying Missing and Empty Values

In [12]:
missing_summary = pd.DataFrame({
    "Column": df.columns,
    "Missing Values": df.isna().sum().values,
    "Empty or Whitespace Values": [
        df[col].str.strip().eq("").sum()
        for col in df.columns
    ]
})
display(missing_summary)

,Column,Missing Values,Empty or Whitespace Values
0,flags,0,0
1,utterance,0,0
2,category,0,0
3,intent,0,0


#### Observations
- **No missing values** were identified across any of the four columns.
- **No empty or whitespace-only entries** were found, confirming that all records contain populated fields.
- **Decision:** No missing-value imputation or empty-record removal is required during data preparation.

#### Examining Utterance Length Statistics

In [13]:
utterance_length_summary = pd.DataFrame({
    "Character Count": df["utterance"].str.len(),
    "Word Count": df["utterance"].str.split().str.len()
})
display(utterance_length_summary.describe().round(2))

,Character Count,Word Count
count,21534.00,21534.00
mean,52.54,10.86
std,16.79,3.72
min,10.00,2.00
25%,41.00,8.00
50%,51.00,10.00
75%,61.00,13.00
max,128.00,28.00


#### Observations
- Customer utterances average **52.54 characters and 10.86 words**, indicating relatively short, conversational queries.
- Utterance lengths range from **2 to 28 words**, with 50% of queries containing approximately 8–13 words.
- **Decision:** The relatively short text lengths support exploring TF-IDF-based classification models, with word-level and character-level features evaluated during feature engineering.

#### Examining Shortest and Longest Utterances

In [14]:
print("Shortest Utterances:")
display(
    df.assign(Word_Count=df["utterance"].str.split().str.len())
    .nsmallest(5, "Word_Count")[["utterance", "intent", "Word_Count"]]
)
print("\nLongest Utterances:")
display(
    df.assign(Word_Count=df["utterance"].str.split().str.len())
    .nlargest(5, "Word_Count")[["utterance", "intent", "Word_Count"]]
)

Shortest Utterances:


,utterance,intent,Word_Count
1125,can Iregister?,create_account,2
4229,registration issue,registration_problems,2
4231,registration issues,registration_problems,2
4234,registration problem,registration_problems,2
7397,contact someone,contact_human_agent,2



Longest Utterances:


,utterance,intent,Word_Count
18953,I got an error message when I attempted to mak...,payment_issue,28
19262,I got an error message when I tried to make a ...,payment_issue,28
15048,I got an error message when I attempted to mak...,payment_issue,27
15394,I got an error message when I tried to make a ...,payment_issue,27
15460,I got an error when I attempted to make a paym...,payment_issue,27


#### Observations
- **Shortest utterances (2 words)** express direct requests related to account registration and customer support contact.
- **Longest utterances (27–28 words)** primarily describe payment-related issues using more detailed language.
- **Decision:** Preserve meaningful short queries and account for spelling variations during text preprocessing to support reliable intent classification.

#### Examining Text Formatting Characteristics

In [15]:
text_characteristics = pd.DataFrame({
    "Characteristic": [
        "Contains Digits",
        "Contains Special Characters",
        "All Lowercase",
        "Leading or Trailing Whitespace",
        "Multiple Consecutive Spaces"
    ],
    "Count": [
        df["utterance"].str.contains(r"\d", regex=True).sum(),
        df["utterance"].str.contains(r"[^a-zA-Z0-9\s]", regex=True).sum(),
        df["utterance"].str.islower().sum(),
        df["utterance"].ne(df["utterance"].str.strip()).sum(),
        df["utterance"].str.contains(r" {2,}", regex=True).sum()
    ]
})
text_characteristics["Percentage"] = (
    text_characteristics["Count"] / len(df) * 100
).round(2)
display(text_characteristics)

,Characteristic,Count,Percentage
0,Contains Digits,0,0.00
1,Contains Special Characters,16348,75.92
2,All Lowercase,8341,38.73
3,Leading or Trailing Whitespace,7,0.03
4,Multiple Consecutive Spaces,402,1.87


#### Observations
- **75.92% of utterances contain special characters**, while no numerical digits were identified.
- **38.73% of utterances are entirely lowercase**, indicating variation in capitalization across customer queries.
- **Decision:** Standardize whitespace and evaluate case normalization during preprocessing, while preserving punctuation and other potentially meaningful characters until their impact is assessed.

#### Examining Intent-to-Category Mapping

In [16]:
intent_category_mapping = (
    df.groupby("intent")
    .agg(
        Category=("category", lambda x: ", ".join(sorted(x.unique()))),
        Unique_Categories=("category", "nunique"),
        Total_Utterances=("utterance", "count")
    )
    .sort_values(["Category", "intent"])
    .reset_index()
)
display(intent_category_mapping)

,intent,Category,Unique_Categories,Total_Utterances
0,create_account,ACCOUNT,1,2122
1,delete_account,ACCOUNT,1,913
2,edit_account,ACCOUNT,1,133
3,recover_password,ACCOUNT,1,986
4,registration_problems,ACCOUNT,1,130
5,switch_account,ACCOUNT,1,273
6,check_cancellation_fee,CANCELLATION_FEE,1,360
7,contact_customer_service,CONTACT,1,2055
8,contact_human_agent,CONTACT,1,1026
9,delivery_options,DELIVERY,1,360


#### Observations
- **All 27 intents map exclusively to one category each**, confirming a consistent relationship across the 11 customer service categories.
- **ACCOUNT contains the most intents (6)**, followed by ORDER (4) and REFUNDS (3), indicating differences in classification complexity across categories.
- **Modeling Decision:** Use `intent` as the primary prediction target and derive its corresponding category when needed, rather than training with category as an input feature.

## 2. Data Understanding Summary

The Bitext Customer Service dataset contains **21,534 customer utterances** across **11 service categories and 27 intent classes**, providing a foundation for multiclass customer support intent classification.

#### Key Findings
- **Data Quality:** No missing or empty values were identified. The dataset contains 12 fully duplicated records and 21 repeated utterances, with no conflicting intent labels.
- **Class Distribution:** Intent frequencies are substantially imbalanced, ranging from 4,366 `payment_issue` records to only 36 `cancel_order` records.
- **Text Characteristics:** Customer utterances average 10.86 words, with variations in capitalization, punctuation, spelling, and whitespace.
- **Classification Structure:** Each intent maps exclusively to one service category, establishing a consistent category–intent hierarchy.

#### Preprocessing and Modeling Decisions
- Use `utterance` as the input feature and `intent` as the multiclass target variable.
- Remove duplicate utterances and standardize unnecessary whitespace during data preparation.
- Investigate text normalization techniques while preserving meaningful linguistic information.
- Exclude `category` and `flags` from initial predictive features to avoid reliance on metadata unavailable during inference.
- Apply stratified train/test splitting and prioritize **Macro F1-score**, alongside per-class precision and recall, to assess performance across imbalanced intent classes.

These findings establish the basis for exploratory analysis, text preprocessing, feature engineering, and subsequent machine learning development.